# Few-Shot Prototype Matching with DINOv2 Backbone Comparison

Phase 10. Cross-tabulates inference strategies (nearest-neighbor, top-k mean, prototype centroid, prototype medoid, prototype with abstention) across embedding models (texture_baseline, learned_metric, dinov2_base_frozen) at K = 1, 3, 5, 10 on LeaData. Optionally evaluates FabricNET as out-of-domain queries.

**Diagnostic only on public texture data.** LeaData class structure is leather *species* (Cow / Goat / Sheep / Buffalo) and *sample groups within species* — neither is authentic-vs-counterfeit at brand granularity. Required K on real luxury data is likely higher than what these results suggest. Do not extrapolate K values directly.

## Why this phase

LuxAuth's production pipeline is already operationally few-shot: image → embedding → vector search → region scoring against a reference cluster. The **support-set sensitivity curve** — accuracy and variance as K grows — is the most operationally useful single output of this phase, because it tells us how many reference images per region the gold-set capture protocol must collect before scores stabilize.

## Embedding regimes compared

- `texture_baseline_v0.2.0` — hand-crafted LBP + Gabor, 256-d.
- `learned_metric_v0.2.0` — ResNet18 contrastive, 128-d.
- `dinov2_base_frozen_v0.1.0` — DINOv2 ViT-B/14 frozen, CLS token, 768-d.


## Run experiments

From `auth-platform/`:

```bash
# Smoke test (50 episodes, single model)
make few-shot-quick

# Full runs (500 episodes per cell, all three embedding models)
make generate-dinov2-embeddings   # one-time, populates pgvector
make few-shot-full-leadata-itemid
make few-shot-full-leadata-species
make few-shot-full-leadata-ood    # adds FabricNET-as-OOD

# Cross-tabulate the latest runs
make compare-few-shot
```


In [ ]:
import json
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

FS_DIR = Path('../experiments/outputs/few-shot')
summary_paths = sorted(FS_DIR.glob('*/few_shot_summary.json'), key=lambda p: p.stat().st_mtime, reverse=True)
print(f'Found {len(summary_paths)} summary files.')
if not summary_paths:
    print('No results yet. Run `make few-shot-full-leadata-itemid` first.')
else:
    print('Most recent:', summary_paths[0])

In [ ]:
def load_cells(path: Path) -> list[dict]:
    summary = json.loads(path.read_text())
    rows = []
    for cell in summary.get('cells', []):
        for strategy, stats in cell.get('strategies', {}).items():
            rows.append({
                'embedding_model': cell['embedding_model'],
                'label_strategy': cell['label_strategy'],
                'K': cell['K'],
                'strategy': strategy,
                **stats,
            })
    return rows

rows = load_cells(summary_paths[0]) if summary_paths else []
len(rows)

In [ ]:
if rows:
    fig, ax = plt.subplots(figsize=(8, 5))
    by_combo = {}
    for r in rows:
        by_combo.setdefault((r['embedding_model'], r['strategy']), []).append(r)
    for (model, strat), group in sorted(by_combo.items()):
        group_sorted = sorted(group, key=lambda r: r['K'])
        Ks = [r['K'] for r in group_sorted]
        accs = [r['accuracy_mean'] for r in group_sorted]
        ax.plot(Ks, accs, marker='o', label=f'{model} / {strat}')
    ax.set_xlabel('K (shots)')
    ax.set_ylabel('Mean episode accuracy')
    ax.set_title('Support-set stabilization curve')
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()

In [ ]:
if rows:
    fig, ax = plt.subplots(figsize=(8, 5))
    by_combo = {}
    for r in rows:
        by_combo.setdefault((r['embedding_model'], r['strategy']), []).append(r)
    for (model, strat), group in sorted(by_combo.items()):
        group_sorted = sorted(group, key=lambda r: r['K'])
        Ks = [r['K'] for r in group_sorted]
        spread = [r['support_set_variance_p95'] for r in group_sorted]
        ax.plot(Ks, spread, marker='s', label=f'{model} / {strat}')
    ax.set_xlabel('K (shots)')
    ax.set_ylabel('p95 - p05 (support-set variance)')
    ax.set_title('Variance vs K — lower means more stable')
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()

In [ ]:
import csv
if summary_paths:
    abst_csv = summary_paths[0].parent / 'abstention_curves.csv'
    if abst_csv.exists():
        with abst_csv.open() as fh:
            curve_rows = list(csv.DictReader(fh))
        fig, ax = plt.subplots(figsize=(8, 5))
        by_model = {}
        for r in curve_rows:
            if int(r['K']) != 5:
                continue
            by_model.setdefault(r['embedding_model'], []).append(r)
        for model, group in sorted(by_model.items()):
            group_sorted = sorted(group, key=lambda r: float(r['abstention_rate']))
            xs = [float(r['abstention_rate']) for r in group_sorted]
            ys = [float(r['accuracy_after_abstention']) for r in group_sorted]
            ax.plot(xs, ys, marker='.', label=model)
        ax.set_xlabel('Abstention rate')
        ax.set_ylabel('Accuracy among non-abstained')
        ax.set_title('Abstention curve at K=5 (prototype_centroid_with_abstention)')
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
    else:
        print('No abstention curve CSV found.')

## Conclusions template

Fill in after a real run completes:

1. **Minimum useful K** per (model, strategy) — smallest K reaching within 0.02 of asymptotic accuracy.
2. **Diminishing returns point** per (model, strategy) — smallest K where (acc at K+next) - (acc at K) < 0.02.
3. **Cross-model ranking** at fixed K (e.g. K=5) — does DINOv2 dominate? On which class structures (species vs item-id)?
4. **Recommended capture protocol K** — based on the most stable (model, strategy) combination, but **caveated**: real luxury data is finer-grained than LeaData species.
5. **Production candidacy** — does DINOv2 outperform meaningfully enough to warrant Phase 11 shadow integration?

**Caveat — restated, deliberately:** these K values are calibrated on LeaData leather-species discrimination. Authentic-vs-counterfeit Chanel caviar 2022 is a finer-grained distinction; required K on real lab data is likely **higher**, not the same.